#### Mix Vidéo (cany) piloté par le Volume Audio

In [ ]:
import cv2
import numpy as np
import pyaudio
import sys

# --- CONFIGURATION AUDIO ---
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 44100

p = pyaudio.PyAudio()


stream = p.open(format=FORMAT,
                channels=CHANNELS,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK)

# --- CONFIGURATION VIDÉO ---
cap = cv2.VideoCapture(0)


while True:
    ret, frame = cap.read() # ret = True or False, frame = image
    if not ret:
        break

    # 1. CALCUL DE L'INTENSITÉ SONORE SÉCURISÉ
    try:
        # Lecture du flux audio
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        
        # Calcul du RMS (Root Mean Square)
        if len(data) > 0:
            rms = np.sqrt(np.mean(data.astype(np.float32)**2))
            
        # On calcule le volume seulement si rms est un nombre valide, sinon 0
        volume = rms / 1000 if np.isfinite(rms) else 0

    except Exception:
        volume = 0
    
    # Alpha contrôle le mixage Dry/Wet (entre 0.0 et 1.0)
    alpha = np.clip(volume, 0.0, 1.0)

    # 2. TRAITEMENT D'IMAGE (CANNY)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Les seuils de Canny s'adaptent au volume pour plus de détails quand il y a du bruit
    low_thresh = 50
    high_thresh = int(150 * (1 - alpha))
    edges = cv2.Canny(blurred, low_thresh, high_thresh)

    edges_colored = np.zeros_like(frame)
    edges_colored[edges > 0] = [0, 255, 0]

    # 4. MIXAGE FINAL (Dry/Wet)
    # Plus il y a de son, plus l'image Canny (Wet) prend le dessus sur l'original (Dry)
    combined = cv2.addWeighted(frame, 1 - alpha, edges_colored, alpha, 0)
    
    # 5. AFFICHAGE DES INFOS
    cv2.putText(combined, f"Audio Level: {alpha:.2f}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow('Filtre Canny Reactif au Son', combined)

    # Quitter avec la touche 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- NETTOYAGE ---
cap.release()
stream.stop_stream()
stream.close()
p.terminate()
cv2.destroyAllWindows()

In [8]:
import cv2
import numpy as np
import pyaudio
import sys

# --- CONFIGURATION AUDIO ---
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 44100

p = pyaudio.PyAudio()

try:
    stream = p.open(format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    frames_per_buffer=CHUNK)
except Exception as e:
    print(f"Erreur Audio : Impossible d'ouvrir le micro. Vérifiez vos paramètres Windows. {e}")
    sys.exit()

# --- CONFIGURATION VIDÉO ---
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erreur Vidéo : Impossible d'accéder à la webcam.")
    sys.exit()

print("Application lancée. Appuyez sur 'q' pour quitter.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. CALCUL DE L'INTENSITÉ SONORE SÉCURISÉ
    try:
        # Lecture du flux audio
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        
        # Calcul du RMS (Root Mean Square)
        if len(data) > 0:
            rms = np.sqrt(np.mean(data.astype(np.float32)**2))
            
            # Protection contre NaN ou Inf
            if np.isnan(rms) or np.isinf(rms):
                volume = 0
            else:
                # Ajustez le diviseur (1000) pour changer la sensibilité
                volume = rms / 1000 
        else:
            volume = 0
    except Exception:
        volume = 0
    
    # Alpha contrôle le mixage Dry/Wet (entre 0.0 et 1.0)
    alpha = np.clip(volume, 0.0, 1.0)

    # 2. TRAITEMENT D'IMAGE (CANNY)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Les seuils de Canny s'adaptent au volume pour plus de détails quand il y a du bruit
    low_thresh = 50
    high_thresh = int(150 * (1 - alpha))
    edges = cv2.Canny(blurred, low_thresh, high_thresh)

    edges_colored = np.zeros_like(frame)
    edges_colored[edges > 0] = [0, 255, 0]

    # 4. MIXAGE FINAL (Dry/Wet)
    # Plus il y a de son, plus l'image Canny (Wet) prend le dessus sur l'original (Dry)
    combined = cv2.addWeighted(frame, 1 - alpha, edges_colored, alpha, 0)
    
    # 5. AFFICHAGE DES INFOS
    cv2.putText(combined, f"Audio Level: {alpha:.2f}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow('Filtre Canny Reactif au Son', combined)

    # Quitter avec la touche 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- NETTOYAGE ---
cap.release()
stream.stop_stream()
stream.close()
p.terminate()
cv2.destroyAllWindows()

Application lancée. Appuyez sur 'q' pour quitter.


#### Motion Detection (Soustraction de fond) --> BOF

In [9]:
import cv2

cap = cv2.VideoCapture(0)
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)

while True:
    ret, frame = cap.read()
    if not ret: 
        break

    # Appliquer le masque de mouvement
    fgmask = fgbg.apply(frame)

    # Nettoyage du bruit (érosion/dilatation)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)

    # Trouver les contours du mouvement
    contours, _ = cv2.findContours(fgmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    for contour in contours:
        if cv2.contourArea(contour) < 500: # Ignore les petits mouvements (bruit)
            continue
        (x, y, w, h) = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow('Motion Detection', frame)
    cv2.imshow('Mask', fgmask)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### Analyse audio de la voie

In [ ]:
import cv2
import numpy as np
import pyaudio
import sys

# --- CONFIGURATION ---
CHUNK = 1024
RATE = 12000
DURATION_SEC = 5
MAX_ROWS = int((RATE / CHUNK) * DURATION_SEC) 
FFT_SIZE = CHUNK // 2 + 1 
BETA = 0.92  # Facteur d'oubli (0.9 = lent/stable, 0.5 = rapide)

# --- INITIALISATION AUDIO ---
p = pyaudio.PyAudio()
try:
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=RATE, 
                    input=True, frames_per_buffer=CHUNK)
except Exception as e:
    print(f"Erreur Audio : {e}")
    sys.exit()

# --- INITIALISATION DES BUFFERS ---
spectro_fifo = np.zeros((MAX_ROWS, FFT_SIZE, 3), dtype=np.uint8)
hanning_window = np.hanning(CHUNK)
# Initialisation du spectre moyen (même taille que la FFT)
mean_spectrum = np.zeros(FFT_SIZE, dtype=np.float32)

print(f"Système lancé. Jaune = Instantané, Rouge = Moyen ({BETA})")

while True:
    try:
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        if data.size < CHUNK: continue

        # 1. FFT ET NORMALISATION
        fft_data = np.abs(np.fft.rfft(data.astype(np.float32) * hanning_window))
        fft_log = 20 * np.log10(fft_data + 1e-6)
        fft_norm = np.clip((fft_log - 20) / 60, 0, 1)
        
        # 2. CALCUL DU SPECTRE MOYEN (EMA)
        # mean = (1-BETA)*instantané + BETA*mean
        mean_spectrum = (1 - BETA) * fft_norm + BETA * mean_spectrum

        # 3. MISE À JOUR SPECTROGRAMME (FIFO)
        spectro_fifo[1:] = spectro_fifo[:-1]
        line_color = cv2.applyColorMap((fft_norm * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
        spectro_fifo[0] = line_color.reshape(FFT_SIZE, 3)

        # 4. CONSTRUCTION DE LA VUE DU SPECTRE (POLYLINES)
        spectre_view = np.zeros((300, 800, 3), dtype=np.uint8)
        
        def get_pts(spectrum, width, height):
            n = len(spectrum)
            pts = []
            for i, v in enumerate(spectrum):
                x = int(i * (width / n))
                y = int(height - (v * height))
                pts.append([x, y])
            return np.array(pts, np.int32).reshape((-1, 1, 2))

        # Tracé du spectre moyen (Rouge)
        pts_mean = get_pts(mean_spectrum, 800, 300)
        cv2.polylines(spectre_view, [pts_mean], False, (0, 0, 200), 2, cv2.LINE_AA)
        
        # Tracé du spectre instantané (Jaune)
        pts_inst = get_pts(fft_norm, 800, 300)
        cv2.polylines(spectre_view, [pts_inst], False, (0, 255, 255), 1, cv2.LINE_AA)

        # 5. ASSEMBLAGE FINAL
        spectro_view = cv2.resize(spectro_fifo, (800, 400), interpolation=cv2.INTER_LINEAR)
        final_display = np.vstack((spectre_view, spectro_view)) # Correction: spectre_view au dessus
        
        # On remet dans l'ordre : Spectre en haut, Spectrogramme en bas
        combined = np.vstack((spectre_view, spectro_view))

        cv2.imshow('Analyseur Audio Expert', combined)

    except Exception as e:
        print(f"Erreur : {e}")
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

stream.stop_stream()
stream.close()
p.terminate()
cv2.destroyAllWindows()

Système lancé. Jaune = Instantané, Rouge = Moyen (0.92)
